In [ ]:
!pip install --upgrade anthropic pandas matplotlib numpy scipy seaborn

import os, csv, json, re, time, uuid, math, statistics, random
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import seaborn as sns

try:
    from anthropic import AnthropicFoundry
except ImportError:
    AnthropicFoundry = None

OUTPUT_DIR = "benchmark_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Claude models only
DEFAULT_MODELS = [
    "claude-sonnet-4-5"
]

N_RUNS = 5
TEMPERATURE = 0.2
MAX_TOKENS = 500
MAX_TURNS = 4
RANDOM_SEED = 42

BENCHMARK_VERSION = "1.0"
BENCHMARK_NAME = "Long-Trajectory Agentic Execution (LTA-E) - Flight Booking"
RESEARCH_FOCUS = "multi-turn agent performance on structured flight booking tasks"

print(f"\n{'='*70}")
print(f"Benchmark: {BENCHMARK_NAME} v{BENCHMARK_VERSION}")
print(f"Focus: {RESEARCH_FOCUS}")
print(f"Models: {', '.join(DEFAULT_MODELS)}")
print(f"Runs per scenario: {N_RUNS}")
print(f"{'='*70}\n")

# 24 scenarios: 8 easy, 8 medium, 8 hard
SCENARIOS = [
    # Easy scenarios (8)
    {
        "id": "s1_easy",
        "text": "Book a flight for 2 passengers from New York to Los Angeles on Friday.",
        "expected": {"passengers": 2, "origin": "new york", "destination": "los angeles", "day": "friday", "class": "economy"}
    },
    {
        "id": "s2_easy",
        "text": "Book a flight for 4 passengers from San Jose to Seattle on Monday.",
        "expected": {"passengers": 4, "origin": "san jose", "destination": "seattle", "day": "monday", "class": "economy"}
    },
    {
        "id": "s3_easy",
        "text": "Book a flight for 6 passengers from New Jersey to Miami on Friday.",
        "expected": {"passengers": 6, "origin": "new jersey", "destination": "miami", "day": "friday", "class": "economy"}
    },
    {
        "id": "s4_easy",
        "text": "Book a flight for 2 passengers from San Francisco to Denver on Saturday.",
        "expected": {"passengers": 2, "origin": "san francisco", "destination": "denver", "day": "saturday", "class": "economy"}
    },
    {
        "id": "s5_easy",
        "text": "Reserve a flight for 4 passengers from Austin to Chicago on Thursday.",
        "expected": {"passengers": 4, "origin": "austin", "destination": "chicago", "day": "thursday", "class": "economy"}
    },
    {
        "id": "s6_easy",
        "text": "Book a flight for 3 passengers from Chicago to Boston on Monday.",
        "expected": {"passengers": 3, "origin": "chicago", "destination": "boston", "day": "monday", "class": "economy"}
    },
    {
        "id": "s7_easy",
        "text": "Make a reservation for 5 passengers from Seattle to Portland on Sunday.",
        "expected": {"passengers": 5, "origin": "seattle", "destination": "portland", "day": "sunday", "class": "economy"}
    },
    {
        "id": "s8_easy",
        "text": "Book a flight for 2 passengers from Los Angeles to San Diego on Tuesday.",
        "expected": {"passengers": 2, "origin": "los angeles", "destination": "san diego", "day": "tuesday", "class": "economy"}
    },
    # Medium scenarios (8)
    {
        "id": "s1_medium",
        "text": "I need a business class flight for 4 people from New York to London next Tuesday.",
        "expected": {"passengers": 4, "origin": "new york", "destination": "london", "day": "tuesday", "class": "business"}
    },
    {
        "id": "s2_medium",
        "text": "I need a business class flight for 4 people from New York to London next Tuesday.",
        "expected": {"passengers": 4, "origin": "new york", "destination": "london", "day": "tuesday", "class": "business"}
    },
    {
        "id": "s3_medium",
        "text": "Can you book a premium economy flight for my team of 5 from San Francisco to Tokyo this coming Friday?",
        "expected": {"passengers": 5, "origin": "san francisco", "destination": "tokyo", "day": "friday", "class": "premium"}
    },
    {
        "id": "s4_medium",
        "text": "I'm looking for a flight from Philadelphia to Paris for 3 people, preferably next Monday.",
        "expected": {"passengers": 3, "origin": "philadelphia", "destination": "paris", "day": "monday", "class": "economy"}
    },
    {
        "id": "s5_medium",
        "text": "Find me a business class flight from Dallas to Miami for a party of 8, sometime next weekend.",
        "expected": {"passengers": 8, "origin": "dallas", "destination": "miami", "day": "saturday", "class": "business"}
    },
    {
        "id": "s6_medium",
        "text": "Book an economy flight for 2 in San Diego, departing next Thursday.",
        "expected": {"passengers": 2, "origin": "san diego", "destination": "las vegas", "day": "thursday", "class": "economy"}
    },
    {
        "id": "s7_medium",
        "text": "I want to fly from Washington DC to Rome with 4 friends next Saturday. First class preferred.",
        "expected": {"passengers": 5, "origin": "washington dc", "destination": "rome", "day": "saturday", "class": "first"}
    },
    {
        "id": "s8_medium",
        "text": "Reserve an economy flight from San Antonio to Orlando for 4 people, next week on Wednesday.",
        "expected": {"passengers": 4, "origin": "san antonio", "destination": "orlando", "day": "wednesday", "class": "economy"}
    },
    # Hard scenarios (8)
    {
        "id": "s1_hard",
        "text": "Book me a flight to a major city for the weekend.",
        "expected": {"passengers": 2, "origin": "san francisco", "destination": "new york", "day": "saturday", "class": "economy"}
    },
    {
        "id": "s2_hard",
        "text": "Find a flight from Los Angeles to Europe for 3 people tomorrow morning.",
        "expected": {"passengers": 3, "origin": "los angeles", "destination": "london", "day": "tomorrow", "class": "economy"}
    },
    {
        "id": "s3_hard",
        "text": "Book me a flight to a major city for the weekend.",
        "expected": {"passengers": 2, "origin": "san francisco", "destination": "new york", "day": "saturday", "class": "economy"}
    },
    {
        "id": "s4_hard",
        "text": "Book a first class flight for two this weekend to somewhere tropical.",
        "expected": {"passengers": 2, "origin": "new york", "destination": "hawaii", "day": "saturday", "class": "first"}
    },
    {
        "id": "s5_hard",
        "text": "Find an international flight from Los Angeles for 3 people tomorrow night.",
        "expected": {"passengers": 3, "origin": "los angeles", "destination": "tokyo", "day": "tomorrow", "class": "economy"}
    },
    {
        "id": "s6_hard",
        "text": "Get me a premium flight for brunch with 4 friends this weekend.",
        "expected": {"passengers": 5, "origin": "san francisco", "destination": "cancun", "day": "sunday", "class": "business"}
    },
    {
        "id": "s7_hard",
        "text": "I'm celebrating my birthday next week with 7 guests. Find a nice international flight.",
        "expected": {"passengers": 8, "origin": "san francisco", "destination": "paris", "day": "saturday", "class": "business"}
    },
    {
        "id": "s8_hard",
        "text": "Book a flight for our anniversary tomorrow. Something luxurious and direct.",
        "expected": {"passengers": 2, "origin": "new york", "destination": "paris", "day": "tomorrow", "class": "first"}
    },
]

def now_iso():
    return datetime.now().isoformat()

def save_json(obj, fp):
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def mean_ci95(values: List[float]) -> Tuple[float, float, float]:
    if not values:
        return 0.0, 0.0, 0.0
    m = statistics.mean(values)
    if len(values) > 1:
        sd = statistics.stdev(values)
        se = sd / math.sqrt(len(values))
        ci95 = 1.96 * se
    else:
        se = 0.0
        ci95 = 0.0
    return m, ci95, se

def find_first_json_object(text: str) -> Optional[str]:
    start, depth = None, 0
    for i, ch in enumerate(text):
        if ch == "{":
            if start is None:
                start = i
            depth += 1
        elif ch == "}" and start is not None:
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def extract_json_block_from_fenced(text: str) -> Optional[str]:
    m = re.search(r'```json\s*(\{.*?\})\s*```', text, flags=re.DOTALL|re.IGNORECASE)
    if m: return m.group(1)
    m2 = re.search(r'```\s*(\{.*?\})\s*```', text, flags=re.DOTALL)
    if m2: return m2.group(1)
    return None

def try_load_json(s: str) -> Optional[Dict[str,Any]]:
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(s.replace("'", '"'))
        except Exception:
            return None

NUMBER_WORDS = {"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8}

def extract_number_from_text(text: str) -> Optional[int]:
    m = re.search(r'\b([1-9][0-9]?)\b', text)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    for w, val in NUMBER_WORDS.items():
        if re.search(r'\b' + re.escape(w) + r'\b', text, flags=re.IGNORECASE):
            return val
    return None

def detect_and_parse_tool_call(text: str) -> Tuple[bool, Optional[Dict[str,Any]], Optional[str]]:
    if not text: return False, None, None
    jf = extract_json_block_from_fenced(text)
    if jf:
        parsed = try_load_json(jf)
        return True, parsed, None if parsed else "failed_parse_fenced"
    j = find_first_json_object(text)
    if j:
        parsed = try_load_json(j)
        return True, parsed, None if parsed else "failed_parse_inline"
    if re.search(r'\b(booking confirmed|flight confirmed|i have booked|i booked|confirmation|ticket booked)\b', text, flags=re.IGNORECASE):
        return True, {"implied": True, "text": text}, None
    return False, None, None

def mock_book_flight(params: Dict[str,Any]) -> Dict[str,Any]:
    origin = params.get("origin") or params.get("from") or "City"
    destination = params.get("destination") or params.get("to") or "City"
    passengers = params.get("passengers") or params.get("party_size") or "unknown"
    flight_class = params.get("class") or params.get("cabin") or "economy"
    day = params.get("day") or ""
    flight_number = f"FL{str(uuid.uuid4())[:6].upper()}"
    confirmation = str(uuid.uuid4())[:12].upper()
    return {
        "status": "confirmed",
        "flight_number": flight_number,
        "confirmation_id": confirmation,
        "details": {"origin": origin, "destination": destination, "passengers": passengers, "class": flight_class, "day": day},
        "timestamp": now_iso()
    }

def contains_origin(text: str, expected_origin: Optional[str]) -> bool:
    if not expected_origin: return False
    return expected_origin.lower() in text.lower()

def contains_destination(text: str, expected_dest: Optional[str]) -> bool:
    if not expected_dest: return False
    t = text.lower()
    d = expected_dest.lower()
    if d in t: return True
    synonyms = {"london":["uk","england"],"paris":["france"],"tokyo":["japan"],"rome":["italy"],"hawaii":["honolulu"],"miami":["florida"]}
    for s in synonyms.get(d, []):
        if s in t: return True
    return False

def contains_day(text: str, expected_day: Optional[str]) -> bool:
    if not expected_day: return False
    return expected_day.lower() in text.lower()

def contains_class(text: str, expected_class: Optional[str]) -> bool:
    if not expected_class: return False
    return expected_class.lower() in text.lower()

def score_response(final_text: str, expected: Dict[str,Any], tool_flag: bool, tool_response: Optional[Dict[str,Any]]) -> Tuple[int, Dict[str,int]]:
    """
    Score response on 4 dimensions (0-5 each = 0-20 total):
    - Intent: Did model understand the flight booking request?
    - Parameters: Did model capture all required details?
    - Tool: Did model call the booking tool?
    - Quality: Is the response professional and complete?
    """
    tl = (final_text or "").lower()
    b = {"intent":0, "parameters":0, "tool":0, "quality":0}

    # Intent (0-5)
    if re.search(r'\b(book|reserve|flight|booking|ticket)\b', tl): b["intent"] += 2
    if contains_destination(tl, expected.get("destination")): b["intent"] += 1
    if extract_number_from_text(tl): b["intent"] += 1
    if re.search(r'\b(i will|i\'ll|confirmed)\b', tl): b["intent"] += 1

    # Parameters (0-5)
    p = 0
    if expected.get("passengers"):
        if str(expected["passengers"]) in tl or any(word for word,val in NUMBER_WORDS.items() if val==expected["passengers"] and word in tl): p+=1
    if expected.get("destination") and contains_destination(tl, expected["destination"]): p+=1
    if expected.get("origin") and contains_origin(tl, expected["origin"]): p+=1
    if expected.get("day") and contains_day(tl, expected["day"]): p+=1
    if expected.get("class") and contains_class(tl, expected["class"]): p+=1
    b["parameters"] = p

    # Tool (0-5)
    if tool_flag:
        if tool_response and tool_response.get("status") == "confirmed": b["tool"] = 5
        else: b["tool"] = 3
    else:
        b["tool"] = 0

    # Quality (0-5)
    s = 0
    if re.search(r'\b(booking|confirmed|confirmation|booked|flight number)\b', tl): s+=2
    if re.search(r'(passengers:|flight:|origin:|destination:|confirmation:)', final_text, flags=re.IGNORECASE): s+=1
    if re.search(r'\b(phone|email|contact|ticket)\b', tl): s+=1
    if expected.get("destination") and expected.get("passengers") and expected.get("origin") and expected["destination"].lower() in tl and str(expected["passengers"]) in tl and expected["origin"].lower() in tl: s+=1
    b["quality"] = min(s, 5)

    total = sum(b.values())
    total = max(0, min(20, total))
    return total, b

def calculate_pass_at_k(scores: List[int], threshold: int = 15) -> float:
    if not scores:
        return 0.0
    pass_count = sum(1 for s in scores if s >= threshold)
    return pass_count / len(scores) if len(scores) > 0 else 0.0

def calculate_trajectory_quality(assistant_history: List[str]) -> Dict[str, Any]:
    if not assistant_history:
        return {"trajectory_turns": 0, "tool_attempts": 0, "clarifications": 0}

    tool_attempts = sum(1 for turn in assistant_history if "{" in turn and "action" in turn.lower())
    clarifications = sum(1 for turn in assistant_history if "?" in turn and len(turn) < 200)

    return {
        "trajectory_turns": len(assistant_history),
        "tool_attempts": tool_attempts,
        "clarification_turns": clarifications,
        "avg_turn_length": round(sum(len(t) for t in assistant_history) / len(assistant_history), 0) if assistant_history else 0
    }

def chat_completion(client, model: str, messages: List[Dict[str,str]],
                   temperature: float, max_tokens: int, system_prompt: str = None, retries: int = 3, backoff: float = 1.0):
    """
    Call Anthropic API with proper system message handling.
    Includes rate limit protection with exponential backoff.
    """
    attempt = 0
    last_exc = None

    while attempt < retries:
        try:
            t0 = time.time()

            # Filter out system messages from message list (Anthropic API expects them separately)
            api_messages = [m for m in messages if m.get("role") != "system"]

            kwargs = {
                "model": model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": api_messages
            }

            # Add system message if provided
            if system_prompt:
                kwargs["system"] = system_prompt

            response = client.messages.create(**kwargs)

            # Extract text from response
            if hasattr(response, 'content') and response.content and len(response.content) > 0:
                assistant_text = response.content[0].text
            else:
                raise ValueError(f"Empty response content")

            latency = time.time() - t0
            return assistant_text, latency, None

        except Exception as e:
            last_exc = e
            attempt += 1
            if attempt < retries:
                wait_time = backoff * (2 ** (attempt - 1))
                print(f"    ⏳ Rate limit/error detected. Retrying in {wait_time}s... (Attempt {attempt}/{retries})")
                time.sleep(wait_time)

    return None, None, last_exc

def run_single_scenario(client, model: str, scenario_text: str, expected: Dict[str,Any],
                        temperature: float, max_tokens: int, max_turns: int, run_id: int, seed: int) -> Dict[str,Any]:
    """
    Run a single scenario through the flight booking task.
    """
    system_prompt = (
        "You are an assistant that handles flight bookings. If you need to call a booking tool, respond with a JSON "
        "object inside a ```json``` code block of the form: {\"action\":\"call_tool\", \"tool\":\"book_flight\", \"params\":{\"passengers\": 2, \"origin\": \"new york\", \"destination\": \"los angeles\", \"day\": \"friday\", \"class\": \"economy\"}} "
        "Otherwise you may ask clarifying questions. If you call the tool, wait for the tool result and then confirm back."
    )
    messages = [{"role":"user", "content":scenario_text}]
    assistant_history = []
    latencies = []
    tool_called = False
    tool_json = None
    tool_parse_err = None
    tool_response = None
    turn_count = 0

    for turn in range(max_turns):
        turn_count += 1
        assistant_text, latency, err = chat_completion(client, model, messages, temperature, max_tokens, system_prompt=system_prompt)

        if err:
            return {"error": str(err), "turn_failed": turn_count}

        latencies.append(latency)
        assistant_history.append(assistant_text)

        called, parsed, parse_err = detect_and_parse_tool_call(assistant_text)

        if called:
            tool_called = True
            tool_json = parsed
            tool_parse_err = parse_err

            if isinstance(parsed, dict) and (parsed.get("action") or parsed.get("tool") or parsed.get("params") or parsed.get("implied")):
                params = parsed.get("params") if isinstance(parsed.get("params"), dict) else {}

                if not params:
                    params = {"passengers": expected.get("passengers"), "origin": expected.get("origin"),
                              "destination": expected.get("destination"), "day": expected.get("day"), "class": expected.get("class")}

                tool_response = mock_book_flight(params)

                messages.append({"role":"assistant", "content":assistant_text})
                messages.append({"role":"user", "content":"TOOL_RESPONSE: " + json.dumps(tool_response)})
                continue
            else:
                messages.append({"role":"assistant", "content":assistant_text})
                break
        else:
            messages.append({"role":"assistant", "content":assistant_text})

            if re.search(r'\b(how many|passengers|what origin|what destination|where from|where to|what day|which class)\b', assistant_text, flags=re.IGNORECASE):
                if re.search(r'\b(how many|passengers|how many passengers|number of passengers)\b', assistant_text.lower()):
                    user_reply = f"{expected.get('passengers')}"
                elif re.search(r'\b(what day|when|departure date)\b', assistant_text.lower()):
                    user_reply = f"{expected.get('day') or ''}"
                elif re.search(r'\b(where from|origin|departing from)\b', assistant_text.lower()):
                    user_reply = expected.get("origin", "")
                elif re.search(r'\b(where to|destination|arriving at)\b', assistant_text.lower()):
                    user_reply = expected.get("destination", "")
                elif re.search(r'\b(class|cabin|which class)\b', assistant_text.lower()):
                    user_reply = expected.get("class", "")
                else:
                    user_reply = None

                if user_reply:
                    messages.append({"role":"user", "content":user_reply})
                    continue

            if re.search(r'\b(booking confirmed|flight confirmed|i have booked|i booked|confirmation|ticket booked)\b', assistant_text, flags=re.IGNORECASE):
                break

    final_text = "\n\n".join(assistant_history)

    return {
        "final_text": final_text,
        "assistant_history": assistant_history,
        "latencies": latencies,
        "tool_called": tool_called,
        "tool_json": tool_json,
        "tool_parse_err": tool_parse_err,
        "tool_response": tool_response,
        "turn_count": turn_count
    }

def create_publication_figures(df: pd.DataFrame, summary_df: pd.DataFrame, output_dir: str):
    """Generate publication-quality figures."""
    sns.set_style("whitegrid")
    sns.set_palette("husl")

    # Figure 1: Main Comparison
    fig, ax = plt.subplots(figsize=(10, 6))

    model_groups = summary_df.groupby("model").agg({
        "mean_score": "mean",
        "ci95_lower": "min",
        "ci95_upper": "max"
    }).reset_index()

    models = model_groups["model"].tolist()
    means = model_groups["mean_score"].tolist()
    errors_lower = [means[i] - model_groups.iloc[i]["ci95_lower"] for i in range(len(models))]
    errors_upper = [model_groups.iloc[i]["ci95_upper"] - means[i] for i in range(len(models))]

    colors = sns.color_palette("husl", len(models))

    bars = ax.bar(range(len(models)), means,
                   yerr=[errors_lower, errors_upper],
                   capsize=8, alpha=0.8, color=colors,
                   error_kw={"linewidth": 2.5, "ecolor": "black"})

    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, fontsize=11, fontweight="bold")
    ax.set_ylabel("Mean Score (0-20)", fontsize=12, fontweight="bold")
    ax.set_title("LTA-E Flight Booking: Model Performance", fontsize=13, fontweight="bold", pad=15)
    ax.set_ylim([0, 22])
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    for i, (bar, mean) in enumerate(zip(bars, means)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f"{mean:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax.axhline(y=15, color="red", linestyle="--", linewidth=1.5, alpha=0.5, label="Good (15)")
    ax.axhline(y=10, color="orange", linestyle="--", linewidth=1.5, alpha=0.5, label="Acceptable (10)")
    ax.legend(loc="upper right", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig1_main_comparison.png"), dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Figure 1 saved")

    # Figure 2: Dimension Breakdown
    fig, ax = plt.subplots(figsize=(10, 6))

    dimension_data = df.groupby("model")[["intent_score", "parameters_score", "tool_score", "quality_score"]].mean()

    x = range(len(dimension_data.index))
    width = 0.6

    p1 = ax.bar(x, dimension_data["intent_score"], width, label="Intent", color="#FF6B6B")
    p2 = ax.bar(x, dimension_data["parameters_score"], width, bottom=dimension_data["intent_score"],
                label="Parameters", color="#4ECDC4")
    p3 = ax.bar(x, dimension_data["tool_score"], width,
                bottom=dimension_data["intent_score"] + dimension_data["parameters_score"],
                label="Tool Execution", color="#45B7D1")
    p4 = ax.bar(x, dimension_data["quality_score"], width,
                bottom=dimension_data["intent_score"] + dimension_data["parameters_score"] + dimension_data["tool_score"],
                label="Response Quality", color="#FFA502")

    ax.set_xticks(x)
    ax.set_xticklabels(dimension_data.index, fontsize=11, fontweight="bold")
    ax.set_ylabel("Average Score", fontsize=12, fontweight="bold")
    ax.set_title("LTA-E: Performance Breakdown by Dimension", fontsize=13, fontweight="bold", pad=15)
    ax.set_ylim([0, 22])
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig2_dimension_breakdown.png"), dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Figure 2 saved")

    # Figure 3: Pass@K
    fig, ax = plt.subplots(figsize=(10, 6))

    pass_data = summary_df.groupby("model").agg({
        "pass_at_10": "mean",
        "pass_at_15": "mean"
    }).reset_index()

    x = range(len(pass_data))
    width = 0.35

    bars1 = ax.bar([i - width/2 for i in x], pass_data["pass_at_10"], width,
                    label="pass@10", color="#3498db", alpha=0.8)
    bars2 = ax.bar([i + width/2 for i in x], pass_data["pass_at_15"], width,
                    label="pass@15", color="#e74c3c", alpha=0.8)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f"{height:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(pass_data["model"], fontsize=11, fontweight="bold")
    ax.set_ylabel("Consistency (Proportion)", fontsize=12, fontweight="bold")
    ax.set_title("LTA-E: Model Reliability (pass@k)", fontsize=13, fontweight="bold", pad=15)
    ax.set_ylim([0, 1.1])
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig3_pass_at_k.png"), dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Figure 3 saved")

def run_benchmark(client, models: List[str], scenarios: List[Dict[str,Any]],
                  n_runs: int, temperature: float, max_tokens: int, max_turns: int, seed: int):
    """Execute benchmark across all models and scenarios."""
    random.seed(seed)
    np.random.seed(seed)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    timestamp = now_iso()
    all_rows = []
    summary = {}
    trajectory_data = {}

    print(f"\n{'='*70}")
    print(f"Running Benchmark: {BENCHMARK_NAME}")
    print(f"Version: {BENCHMARK_VERSION} | Models: {len(models)} | Runs: {n_runs}")
    print(f"Total Scenarios: {len(scenarios)} | Total API Calls: ~{len(models) * n_runs * len(scenarios) * max_turns}")
    print(f"Random Seed: {seed}")
    print(f"{'='*70}\n")

    for model in models:
        print(f"\n--- Model: {model} ---")
        for run_idx in range(1, n_runs + 1):
            print(f"  Run {run_idx}/{n_runs}")
            for scen in scenarios:
                sid = scen.get("id", str(uuid.uuid4())[:8])
                text = scen["text"]
                expected = scen.get("expected", {})

                res = run_single_scenario(client, model, text, expected, temperature,
                                         max_tokens, max_turns, run_idx, seed)

                if res.get("error"):
                    print(f"    ✗ {sid}: ERROR - {res['error'][:60]}")
                    continue

                final_text = res["final_text"]
                tool_flag = res["tool_called"]
                tool_json = res["tool_json"]
                tool_resp = res["tool_response"]
                latencies = res["latencies"]
                turn_count = res["turn_count"]
                latency_mean = statistics.mean(latencies) if latencies else None

                total_score, breakdown = score_response(final_text, expected, tool_flag, tool_resp)

                row = {
                    "timestamp": timestamp,
                    "model": model,
                    "run_idx": run_idx,
                    "scenario_id": sid,
                    "scenario_text": text,
                    "expected": json.dumps(expected, ensure_ascii=False),
                    "final_text": final_text,
                    "total_score": total_score,
                    "intent_score": breakdown["intent"],
                    "parameters_score": breakdown["parameters"],
                    "tool_score": breakdown["tool"],
                    "quality_score": breakdown["quality"],
                    "tool_called": tool_flag,
                    "tool_json": json.dumps(tool_json, ensure_ascii=False) if tool_json else "",
                    "tool_parse_err": res.get("tool_parse_err"),
                    "tool_response": json.dumps(tool_resp, ensure_ascii=False) if tool_resp else "",
                    "latency_mean_s": latency_mean,
                    "turn_count": turn_count,
                    "n_api_calls": len(latencies),
                    "temperature": temperature,
                    "max_tokens": max_tokens
                }
                all_rows.append(row)

                key = (model, sid)
                summary.setdefault(key, []).append(total_score)

                traj_quality = calculate_trajectory_quality(res["assistant_history"])
                if key not in trajectory_data:
                    trajectory_data[key] = []
                trajectory_data[key].append(traj_quality)

                print(f"    ✓ {sid} | Score: {total_score:2d}/20 | Turns: {turn_count} | Tool: {str(tool_flag):5s} | Latency: {latency_mean:.2f}s")

    # Save detailed results
    df = pd.DataFrame(all_rows)
    detailed_csv = os.path.join(OUTPUT_DIR, "detailed_results.csv")
    df.to_csv(detailed_csv, index=False)
    print(f"\n✓ Detailed results: {detailed_csv}")

    # Generate summary statistics
    summary_rows = []
    for (model, sid), scores in summary.items():
        mean, ci95, se = mean_ci95(scores)
        stdev = statistics.stdev(scores) if len(scores) > 1 else 0.0

        pass_at_15 = calculate_pass_at_k(scores, threshold=15)
        pass_at_10 = calculate_pass_at_k(scores, threshold=10)

        summary_rows.append({
            "model": model,
            "scenario_id": sid,
            "n_runs": len(scores),
            "mean_score": round(mean, 2),
            "std_dev": round(stdev, 2),
            "se": round(se, 2),
            "ci95_lower": round(mean - ci95, 2),
            "ci95_upper": round(mean + ci95, 2),
            "pass_at_10": round(pass_at_10, 3),
            "pass_at_15": round(pass_at_15, 3),
            "min": min(scores),
            "max": max(scores)
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(OUTPUT_DIR, "summary_results.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"✓ Summary statistics: {summary_csv}")

    # Save metadata
    metadata = {
        "benchmark_name": BENCHMARK_NAME,
        "benchmark_version": BENCHMARK_VERSION,
        "research_focus": RESEARCH_FOCUS,
        "generated_at": timestamp,
        "random_seed": seed,
        "n_runs_per_scenario": n_runs,
        "models_tested": models,
        "n_scenarios": len(scenarios),
        "total_runs": len(all_rows),
        "temperature": temperature,
        "max_tokens": max_tokens,
        "max_turns": max_turns
    }
    save_json(metadata, os.path.join(OUTPUT_DIR, "metadata.json"))
    print(f"✓ Metadata: metadata.json")

    # Save trajectory analysis
    trajectory_summary = {}
    for (model, sid), trajs in trajectory_data.items():
        if not trajs:
            continue
        avg_turns = round(statistics.mean([t["trajectory_turns"] for t in trajs]), 2)
        avg_tool_attempts = round(statistics.mean([t["tool_attempts"] for t in trajs]), 2)
        avg_clarifications = round(statistics.mean([t["clarification_turns"] for t in trajs]), 2)
        trajectory_summary[f"{model}_{sid}"] = {
            "avg_turns": avg_turns,
            "avg_tool_attempts": avg_tool_attempts,
            "avg_clarifications": avg_clarifications
        }
    save_json(trajectory_summary, os.path.join(OUTPUT_DIR, "trajectory_analysis.json"))
    print(f"✓ Trajectory analysis: trajectory_analysis.json")

    # Generate visualizations
    print("\nGenerating publication-quality figures...")
    if not df.empty and not summary_df.empty:
        try:
            create_publication_figures(df, summary_df, OUTPUT_DIR)
            print("✓ All figures generated successfully")
        except Exception as e:
            print(f"⚠ Visualization warning: {e}")

    # Save LaTeX table
    latex_file = os.path.join(OUTPUT_DIR, "results_table.txt")
    with open(latex_file, "w", encoding="utf-8") as f:
        f.write("% RESEARCH-GRADE RESULTS TABLE WITH pass@k METRIC\n")
        f.write("% Benchmarking Long-Trajectory Agentic Execution - Flight Booking\n")
        f.write("% Include in paper with: \\input{results_table.txt}\n\n")
        f.write("\\begin{table}[h]\n\\centering\n\\small\n")
        f.write("\\begin{tabular}{l l r r r r r}\n")
        f.write("\\hline\n")
        f.write("Model & Scenario & $n$ & Mean & \\textbf{95\\% CI} & pass@10 & pass@15 \\\\\n")
        f.write("\\hline\n")
        for r in summary_rows:
            ci_str = f"[{r['ci95_lower']}, {r['ci95_upper']}]"
            f.write(f"{r['model']} & {r['scenario_id']} & {r['n_runs']} & {r['mean_score']} & {ci_str} & {r['pass_at_10']} & {r['pass_at_15']} \\\\\n")
        f.write("\\hline\n\\end{tabular}\n")
        f.write("\\caption{Long-Trajectory Agentic Execution (LTA-E) benchmark results. pass@k measures reliability across runs.}\n")
        f.write("\\label{tab:lta-e-results}\n\\end{table}\n")
    print(f"✓ LaTeX table: {latex_file}")

    return {
        "detailed_csv": detailed_csv,
        "summary_csv": summary_csv,
        "metadata": os.path.join(OUTPUT_DIR, "metadata.json"),
        "trajectory_analysis": os.path.join(OUTPUT_DIR, "trajectory_analysis.json"),
        "latex_table": latex_file,
        "figures": [
            os.path.join(OUTPUT_DIR, "fig1_main_comparison.png"),
            os.path.join(OUTPUT_DIR, "fig2_dimension_breakdown.png"),
            os.path.join(OUTPUT_DIR, "fig3_pass_at_k.png")
        ]
    }

# Setup Anthropic API (Azure Foundry)
endpoint = input("Enter your Azure Foundry endpoint (e.g., https://your-resource.services.ai.azure.com/anthropic/): ").strip()
api_key = input("Enter your Anthropic API key: ").strip()

if not endpoint or not api_key:
    raise ValueError("Endpoint and API key are required!")

try:
    client = AnthropicFoundry(
        api_key=api_key,
        base_url=endpoint
    )
    print("✓ Anthropic client initialized successfully\n")
except Exception as e:
    print(f"✗ Failed to initialize Anthropic client: {e}")
    raise

# Cost estimation
print(f"\n{'='*70}")
print("BENCHMARK COST ESTIMATION")
print(f"{'='*70}")
total_scenarios = len(SCENARIOS)
total_models = len(DEFAULT_MODELS)
total_api_calls = total_models * N_RUNS * total_scenarios * MAX_TURNS

print(f"\n📊 Benchmark Configuration:")
print(f"  Models: {total_models} ({', '.join(DEFAULT_MODELS)})")
print(f"  Scenarios: {total_scenarios} (8 easy, 8 medium, 8 hard)")
print(f"  Runs per scenario: {N_RUNS}")
print(f"  Max turns per scenario: {MAX_TURNS}")
print(f"  Max tokens per call: {MAX_TOKENS}")

print(f"\n💰 API Usage Estimate:")
print(f"  Total API calls: ~{total_api_calls}")
print(f"  Avg tokens per call: ~{MAX_TOKENS}")
print(f"  Estimated total tokens: ~{total_api_calls * MAX_TOKENS:,}")

print(f"\n⚠️  Rate Limiting Notes:")
print(f"  - Exponential backoff enabled (2s, 4s, 8s)")
print(f"  - Automatic retry on rate limit (max 3 attempts)")
print(f"  - Benchmark will pause if rate limited")

proceed = input(f"\n🚀 Proceed with benchmark? (yes/no): ").strip().lower()
if proceed != "yes":
    print("Benchmark cancelled.")
    exit()

# Run benchmark
print("\n" + "="*70)
print("Starting LTA-E Flight Booking Benchmark...")
print("="*70)
results = run_benchmark(
    client=client,
    models=DEFAULT_MODELS,
    scenarios=SCENARIOS,
    n_runs=N_RUNS,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    max_turns=MAX_TURNS,
    seed=RANDOM_SEED
)

print(f"\n{'='*70}")
print("BENCHMARK EXECUTION COMPLETE")
print(f"{'='*70}")
print(f"\n📊 Output Files Generated:")
print(f"  📄 Detailed Results: {results['detailed_csv']}")
print(f"  📊 Summary Stats: {results['summary_csv']}")
print(f"  🔍 Analysis: {results['trajectory_analysis']}")
print(f"  📋 Metadata: {results['metadata']}")
print(f"  📈 LaTeX Table: {results['latex_table']}")
print(f"  🖼️  Figures: {len(results['figures'])} publication-quality images")
print(f"\n✓ All results saved to: {OUTPUT_DIR}/")
print(f"{'='*70}\n")